# Semana 7: Regla de la Cadena y Derivadas de Orden Superior
## Notebook Conceptual (NB1) - Backpropagation Manual

### Propósito de la sesión
Descubrir cómo el error final de una red neuronal depende de todos los pesos intermedios, gracias a la **regla de la cadena**. Implementaremos manualmente el **backpropagation** para una red neuronal diminuta (2 entradas, 1 neurona oculta, 1 salida) y verificaremos los resultados con la diferenciación automática de PyTorch.

### Objetivos de aprendizaje
*   Comprender la **regla de la cadena** como base del backpropagation.
*   Definir una red neuronal como composición de funciones.
*   Calcular manualmente los gradientes de la pérdida respecto a todos los pesos.
*   Visualizar el flujo de gradientes a través de la red.
*   Verificar los gradientes calculados manualmente usando **PyTorch** y su diferenciación automática (`autograd`).
*   Conectar estos conceptos con aplicaciones en NLP (BPTT) y RL (Policy Gradient).

## Configuración Inicial

Importamos las librerías necesarias: NumPy para cálculos manuales, PyTorch para diferenciación automática y Matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Verificamos versión de PyTorch
print(f"PyTorch versión: {torch.__version__}")
print(f"¿CUDA disponible? {torch.cuda.is_available()}")

# Fijamos semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\n🎯 Librerías importadas correctamente.")

---
## 1. La Red Neuronal Diminuta

Definimos una red con:
*   2 entradas: $x_1, x_2$
*   1 neurona oculta con activación sigmoide
*   1 neurona de salida con activación sigmoide (para clasificación binaria)

### 1.1. Arquitectura

**Forward pass:**

$$z_1 = w_{11} x_1 + w_{12} x_2 + b_1$$
$$a_1 = \sigma(z_1)$$
$$z_2 = w_{21} a_1 + b_2$$
$$\hat{y} = \sigma(z_2)$$

### 1.2. Función de pérdida (Binary Cross Entropy)

Para un ejemplo con etiqueta $y \in \{0,1\}$:

$$L = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

In [ ]:
# Datos dummy: un solo ejemplo
x = np.array([1.5, -0.5])  # dos entradas
y = 1.0  # etiqueta (clase positiva)

# Inicialización de pesos y sesgos
w11 = 0.5
w12 = -0.3
b1 = 0.1
w21 = 0.8
b2 = -0.2

print("🎯 Datos y pesos iniciales:")
print(f"  x = {x}")
print(f"  y = {y}")
print(f"  w11 = {w11}, w12 = {w12}, b1 = {b1}")
print(f"  w21 = {w21}, b2 = {b2}")

---
## 2. Forward Pass Manual

Implementamos la propagación hacia adelante paso a paso.

In [ ]:
# Función sigmoide
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Forward pass
z1 = w11 * x[0] + w12 * x[1] + b1
a1 = sigmoid(z1)
z2 = w21 * a1 + b2
y_pred = sigmoid(z2)

# Pérdida (Binary Cross Entropy)
loss = - (y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))

print("🔷 Forward pass:")
print(f"  z1 = {z1:.4f}")
print(f"  a1 = {a1:.4f} (activación oculta)")
print(f"  z2 = {z2:.4f}")
print(f"  ŷ = {y_pred:.4f} (predicción)")
print(f"  Loss = {loss:.4f}")

---
## 3. Backward Pass Manual: Aplicando la Regla de la Cadena

Calcularemos los gradientes de la pérdida respecto a todos los parámetros: $\frac{\partial L}{\partial w_{11}}, \frac{\partial L}{\partial w_{12}}, \frac{\partial L}{\partial b_1}, \frac{\partial L}{\partial w_{21}}, \frac{\partial L}{\partial b_2}$.

### 3.1. Derivadas de funciones elementales

Primero, recordemos algunas derivadas útiles:

*   Derivada de la sigmoide: $\sigma'(z) = \sigma(z)(1-\sigma(z))$
*   Derivada de BCE respecto a ŷ: $\frac{\partial L}{\partial \hat{y}} = -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}} = \frac{\hat{y} - y}{\hat{y}(1-\hat{y})}$

In [ ]:
# --- Cálculo de gradientes paso a paso ---

# 1. Gradiente de la pérdida respecto a ŷ
dL_dy = (y_pred - y) / (y_pred * (1 - y_pred))
print(f"dL/dŷ = {dL_dy:.4f}")

# 2. Gradiente respecto a z2 (salida lineal)
# ŷ = σ(z2) => dŷ/dz2 = σ'(z2) = σ(z2)*(1-σ(z2)) = y_pred*(1-y_pred)
dy_dz2 = y_pred * (1 - y_pred)
dL_dz2 = dL_dy * dy_dz2  # regla de la cadena
print(f"dL/dz2 = {dL_dz2:.4f}")

# 3. Gradientes respecto a w21 y b2
# z2 = w21 * a1 + b2
dz2_dw21 = a1
dz2_db2 = 1.0

dL_dw21 = dL_dz2 * dz2_dw21
dL_db2 = dL_dz2 * dz2_db2
print(f"dL/dw21 = {dL_dw21:.4f}")
print(f"dL/db2 = {dL_db2:.4f}")

# 4. Gradiente respecto a a1 (activación oculta)
# z2 = w21 * a1 + b2 => dz2/da1 = w21
dz2_da1 = w21
dL_da1 = dL_dz2 * dz2_da1
print(f"dL/da1 = {dL_da1:.4f}")

# 5. Gradiente respecto a z1
# a1 = σ(z1) => da1/dz1 = σ'(z1) = a1*(1-a1)
da1_dz1 = a1 * (1 - a1)
dL_dz1 = dL_da1 * da1_dz1
print(f"dL/dz1 = {dL_dz1:.4f}")

# 6. Gradientes respecto a w11, w12, b1
# z1 = w11*x1 + w12*x2 + b1
dz1_dw11 = x[0]
dz1_dw12 = x[1]
dz1_db1 = 1.0

dL_dw11 = dL_dz1 * dz1_dw11
dL_dw12 = dL_dz1 * dz1_dw12
dL_db1 = dL_dz1 * dz1_db1

print(f"\n🔷 Gradientes finales:")
print(f"  dL/dw11 = {dL_dw11:.4f}")
print(f"  dL/dw12 = {dL_dw12:.4f}")
print(f"  dL/db1 = {dL_db1:.4f}")
print(f"  dL/dw21 = {dL_dw21:.4f}")
print(f"  dL/db2 = {dL_db2:.4f}")

### 3.2. Visualización del flujo de gradientes

In [ ]:
# Creamos un diagrama simple del flujo de gradientes
fig, ax = plt.subplots(figsize=(12, 6))

# Coordenadas de los nodos
nodes = {
    'x1': (0, 2), 'x2': (0, 1),
    'w11': (1, 2.2), 'w12': (1, 0.8), 'b1': (1, 1.5),
    'z1': (2, 1.5),
    'a1': (3, 1.5),
    'w21': (4, 2), 'b2': (4, 1),
    'z2': (5, 1.5),
    'y_pred': (6, 1.5),
    'loss': (7, 1.5)
}

# Dibujamos nodos
for node, (x, y) in nodes.items():
    ax.plot(x, y, 'o', markersize=15, color='lightblue')
    ax.text(x, y, node, ha='center', va='center', fontsize=10)

# Conexiones forward (negro)
forward_edges = [
    ('x1', 'w11'), ('x2', 'w12'), ('w11', 'z1'), ('w12', 'z1'), ('b1', 'z1'),
    ('z1', 'a1'),
    ('a1', 'w21'), ('w21', 'z2'), ('b2', 'z2'),
    ('z2', 'y_pred'),
    ('y_pred', 'loss')
]

for src, dst in forward_edges:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    ax.arrow(x1, y1, (x2-x1)*0.8, (y2-y1)*0.8,
              head_width=0.1, head_length=0.1, fc='black', ec='black', alpha=0.5)

# Conexiones backward (rojo) - solo algunas para no saturar
backward_edges = [
    ('loss', 'y_pred'),
    ('y_pred', 'z2'),
    ('z2', 'w21'), ('z2', 'b2'),
    ('z2', 'a1'),
    ('a1', 'z1'),
    ('z1', 'w11'), ('z1', 'w12'), ('z1', 'b1')
]

for src, dst in backward_edges:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    ax.arrow(x1, y1, (x2-x1)*0.8, (y2-y1)*0.8,
              head_width=0.1, head_length=0.1, fc='red', ec='red', alpha=0.5, linestyle='dashed')

ax.set_xlim(-0.5, 7.5)
ax.set_ylim(0.5, 2.8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Flujo de Gradientes (Backpropagation): Forward (negro) / Backward (rojo)', fontsize=14)
plt.show()

---
## 4. Verificación con Diferenciación Automática de PyTorch

Ahora implementamos la misma red en PyTorch y usamos `autograd` para calcular los gradientes automáticamente. Esto nos permitirá verificar nuestros cálculos manuales.

In [ ]:
# Definimos los parámetros como tensores con requires_grad=True
w11_t = torch.tensor(0.5, requires_grad=True)
w12_t = torch.tensor(-0.3, requires_grad=True)
b1_t = torch.tensor(0.1, requires_grad=True)
w21_t = torch.tensor(0.8, requires_grad=True)
b2_t = torch.tensor(-0.2, requires_grad=True)

# Datos de entrada
x_t = torch.tensor([1.5, -0.5])
y_t = torch.tensor(1.0)

# Forward pass en PyTorch
z1_t = w11_t * x_t[0] + w12_t * x_t[1] + b1_t
a1_t = torch.sigmoid(z1_t)
z2_t = w21_t * a1_t + b2_t
y_pred_t = torch.sigmoid(z2_t)

# Pérdida (Binary Cross Entropy)
loss_t = - (y_t * torch.log(y_pred_t) + (1 - y_t) * torch.log(1 - y_pred_t))

print(f"🔶 Forward pass con PyTorch:")
print(f"  ŷ = {y_pred_t.item():.4f}")
print(f"  Loss = {loss_t.item():.4f}")

# Backward pass automático
loss_t.backward()

print("\n🔶 Gradientes calculados por PyTorch autograd:")
print(f"  dL/dw11 = {w11_t.grad.item():.4f}")
print(f"  dL/dw12 = {w12_t.grad.item():.4f}")
print(f"  dL/db1 = {b1_t.grad.item():.4f}")
print(f"  dL/dw21 = {w21_t.grad.item():.4f}")
print(f"  dL/db2 = {b2_t.grad.item():.4f}")

In [ ]:
# Comparación con nuestros gradientes manuales
print("📊 Comparación de gradientes (Manual vs PyTorch):")
print(f"{'Parámetro':<10} {'Manual':<15} {'PyTorch':<15} {'Diferencia':<15}")
print("-" * 55)

grads_manual = [dL_dw11, dL_dw12, dL_db1, dL_dw21, dL_db2]
grads_torch = [w11_t.grad.item(), w12_t.grad.item(), b1_t.grad.item(),
               w21_t.grad.item(), b2_t.grad.item()]
names = ['w11', 'w12', 'b1', 'w21', 'b2']

for name, m, t in zip(names, grads_manual, grads_torch):
    diff = abs(m - t)
    print(f"{name:<10} {m:<15.6f} {t:<15.6f} {diff:<15.2e}")

max_diff = max(abs(m - t) for m, t in zip(grads_manual, grads_torch))
print(f"\n✅ Máxima diferencia absoluta: {max_diff:.2e}")
if max_diff < 1e-6:
    print("✅ Los gradientes coinciden perfectamente.")
else:
    print("⚠️ Hay pequeñas diferencias (pueden deberse a precisión numérica).")

---
## 5. Derivadas de Orden Superior y la Matriz Hessiana

Las derivadas de orden superior miden la **curvatura** de la función de pérdida. En optimización, la matriz **Hessiana** (matriz de segundas derivadas) nos indica si estamos en un mínimo, máximo o punto de silla.

### 5.1. Cálculo del Hessiano con PyTorch

Para una función de dos variables, podemos calcular el Hessiano. En nuestro caso, la función de pérdida depende de 5 parámetros, así que el Hessiano sería una matriz 5x5. Aquí calcularemos un ejemplo más simple.

In [ ]:
# Ejemplo simple: f(w) = w^2 (un solo parámetro)
w = torch.tensor(1.0, requires_grad=True)
f = w**2

# Primera derivada
grad_f = torch.autograd.grad(f, w, create_graph=True)[0]
print(f"f'(w) = {grad_f.item()}")

# Segunda derivada (derivada de la primera derivada)
hess_f = torch.autograd.grad(grad_f, w, create_graph=True)[0]
print(f"f''(w) = {hess_f.item()}")

# Para nuestro caso, podríamos calcular el Hessiano completo, pero es computacionalmente costoso.
# En la práctica, se usan aproximaciones (ej. diagonal del Hessiano) para optimizadores como Adam.

print("\n📌 En problemas reales, el Hessiano tiene dimensiones (número de parámetros)²,")
print("   por lo que rara vez se calcula explícitamente. Se usan métodos de segundo orden aproximados.")

---
## 6. Conexiones IA: Aplicaciones de la Regla de la Cadena

### 6.1. Deep Learning: Backpropagation
Como hemos visto, la regla de la cadena permite calcular gradientes en redes profundas.

### 6.2. NLP: Backpropagation Through Time (BPTT)
En RNNs, LSTMs y GRUs, la regla de la cadena se aplica a través de los pasos temporales.

### 6.3. Reinforcement Learning: Policy Gradient
En algoritmos como REINFORCE, la regla de la cadena se usa para actualizar la política.

In [ ]:
# Ejemplo conceptual de BPTT (simplificado)
# Para una RNN con 3 pasos temporales, la pérdida depende de todas las salidas intermedias.
# La regla de la cadena se aplica a lo largo del tiempo, acumulando gradientes.

print("📌 En BPTT, el gradiente total es la suma de los gradientes en cada paso temporal:")
print("   dL/dW = Σ_t dL_t/dW = Σ_t (dL_t/dh_t * dh_t/dh_{t-1} * ... * dh_1/dW)")
print("   Esto es una aplicación directa de la regla de la cadena.")

---
## Ejercicios para el Estudiante

1.  **Cambio de activación:** Recalcula los gradientes manualmente pero usando **ReLU** en lugar de sigmoide en la capa oculta. ¿Qué cambios observas? ¿En qué puntos hay problemas de diferenciabilidad?

2.  **Red con dos neuronas ocultas:** Extiende la red para tener 2 neuronas en la capa oculta. Escribe las ecuaciones del forward pass y deriva las expresiones para los gradientes.

3.  **Verificación con JAX:** (Opcional) Implementa la misma red en JAX y usa `jax.grad` para verificar los gradientes.

4.  **Hessiano aproximado:** Investiga el optimizador **Adam**. ¿Cómo utiliza información de segundo orden sin calcular explícitamente el Hessiano?

5.  **Reflexión:** ¿Por qué el backpropagation es tan eficiente comparado con el cálculo numérico de gradientes (diferencias finitas)? Relaciona con la regla de la cadena.

---
## Conclusiones de la Sesión

*   La **regla de la cadena** es la herramienta matemática que permite descomponer la derivada de una función compuesta en el producto de las derivadas de sus componentes.
*   En una red neuronal, la pérdida es una función compuesta por todas las capas. El **backpropagation** aplica la regla de la cadena para calcular eficientemente los gradientes de todos los parámetros.
*   Hemos implementado manualmente el backpropagation para una red diminuta, verificando cada paso y comparando con la diferenciación automática de PyTorch.
*   Las **derivadas de orden superior** (Hessiano) miden la curvatura de la función de pérdida y son la base de optimizadores de segundo orden.
*   Estos conceptos se extienden a otras áreas de IA como NLP (BPTT) y Reinforcement Learning (Policy Gradient).

¡Ahora entiendes el corazón matemático del entrenamiento de redes neuronales!